# Confronto dei piani di rimborso dei prestiti studenteschi con PROC LOAN

## Riepilogo esecutivo

Un ufficio prestiti studenteschi valuta come una coorte di neolaureati dovrebbe rimborsare un saldo rappresentativo di $27.500 confrontando quattro strutture di rimborso — un piano federale a tasso fisso standard, un rifinanziamento privato con commissione di attivazione, un prestito a tasso variabile con tetto massimo (ARM) e una riduzione del tasso sponsorizzata dal datore di lavoro — utilizzando `PROC LOAN`. Su una durata di 120 mesi le quattro offerte si allineano chiaramente su rata mensile e interessi totali ai loro tassi iniziali dichiarati: il piano federale standard costa di più in interessi (**$10.021,22** al 6,53%, rata **$312,68**), il rifinanziamento privato riduce il tasso ma aggiunge una commissione di $412,50 (**$9.120,20** al 5,99%, rata **$305,17**), mentre l'ARM quotato più basso (**$7.099,76** al 4,75%, rata **$288,33**) e la riduzione tasso del datore di lavoro (**$6.700,67** al 4,50%, rata **$285,01**) comportano gli oneri di interesse minori. Un blocco `COMPARE` riporta poi gli interessi cumulativi, il capitale e il saldo residuo di ciascun piano a 36, 60 e 120 mesi, mostrando quanto ciascun prestito si sia ammortizzato nei punti in cui un mutuatario è più probabile che rifinanzi o estingua il debito.

## Origine dei dati

| Dataset | Righe | Descrizione | Variabili chiave |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Profili sintetici di prestito di una coorte di neolaureati, generati inline con `call streaminit(20260531)` e `rand('uniform')`. Usati per motivare condizioni di prestito realistiche per il confronto. | `student_id` (1001-1040), `balance` (capitale alla laurea; questa estrazione copre $11.800-$47.750, media $30.771), `apr` (tasso nominale annuo 4,10%-9,10%, media 6,50%), `term` (120 o 180 mesi, media 144), `origination_pct` (commissione 1,0%-2,0%, media 1,50%) |

Il mutuatario rappresentativo analizzato con `PROC LOAN` (importo $27.500, durata 120 mesi, avvio luglio 2026) si colloca nella parte medio-bassa della distribuzione di saldo e tasso di questa coorte; non viene utilizzato alcun dato esterno o di rete. La coorte esiste per motivare condizioni di prestito plausibili — il confronto diretto viene eseguito sul singolo prestito rappresentativo.

# Confronto dei piani di rimborso dei prestiti studenteschi con PROC LOAN

Quando gli studenti si laureano, un ufficio prestiti deve aiutarli a scegliere tra offerte di rimborso concorrenti. `PROC LOAN` (SAS/ETS) è progettata esattamente per questo: ammortizza prestiti a tasso fisso, a tasso variabile (ARM) e con riduzione iniziale del tasso, per poi confrontarli fianco a fianco su rata, interessi totali e avanzamento dell'ammortamento.

In questo notebook:

1. Generiamo una coorte sintetica di neolaureati per stabilire condizioni di prestito realistiche.
2. Riepiloghiamo la coorte con `PROC MEANS`.
3. Costruiamo quattro piani di rimborso per un saldo rappresentativo di $27.500 e li confrontiamo con `PROC LOAN` (FIXED / ARM / BUYDOWN + COMPARE).
4. Rieseguiamo il piano fisso raccomandato da solo per confermarne l'economia autonoma.

Tutto viene eseguito localmente su dati sintetici inline — nessun file esterno o accesso di rete.

## Passo 1 — Generare una coorte sintetica di neolaureati

Simuliamo 40 mutuatari. Ciascuno estrae un saldo di capitale alla laurea, un APR legato in modo approssimativo alla qualità creditizia, una durata di rimborso standard (10 o 15 anni) e una percentuale di commissione di attivazione. Il seme rende i dati riproducibili.

In [1]:
DATI borrowers;
   CHIAMARE streaminit(20260531);
   LUNGHEZZA plan $ 28;
   FARE student_id = 1001 FINO_A 1040;
      /* Capitale alla laurea: $9.500 - $48.000 */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Tasso annuo nominale per fascia di rischio: 4,0% - 9,5% */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Durata di rimborso standard: 120 o 180 mesi */
      SE_COND rand('uniform') < 0.6 ALLORA term = 120;
      ALTRIMENTI term = 180;
      /* Commissione di attivazione come percentuale del capitale: 1,0% - 2,0% */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      USCITA;
   FINE;
ESEGUIRE;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Passo 2 — Profilare la coorte

Prima di modellare i singoli prestiti, osserviamo la distribuzione di saldi, tassi e durate. Questo ci dice come appare un mutuatario *rappresentativo* — la base per il confronto diretto che segue.

In [2]:
PROCEDURA MEDIE DATI=borrowers n mean MIN MAX maxdec=2;
   VARIABILE balance apr term origination_pct;
   ETICHETTA balance="Saldo capitale ($)" apr="Tasso annuo nominale (%)"
         term="Durata (mesi)" origination_pct="Commissione attivazione (%)";
ESEGUIRE;

                                                  The MEANS Procedure

 Variable         Label                              N           Mean     Minimum     Maximum
 --------------------------------------------------------------------------------------------
 balance          Saldo capitale ($)                40       30771.25    11800.00    47750.00
 apr              Tasso annuo nominale (%)          40           6.50        4.10        9.10
 term             Durata (mesi)                     40         144.00      120.00      180.00
 origination_pct  Commissione attivazione (%)       40           1.50        1.00        2.00
 --------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Passo 3 — Confrontare quattro piani di rimborso per un mutuatario rappresentativo

Utilizzando un saldo rappresentativo di $27.500 su una durata di 120 mesi (10 anni) con avvio a luglio 2026, allineiamo quattro offerte realistiche:

- **Federale Diretto Non Sovvenzionato (Standard)** — un semplice prestito a tasso fisso al 6,53%.
- **Rifinanziamento Privato (con commissione)** — un tasso fisso più basso al 5,99%, ma con un costo di attivazione di $412,50 (`INIT=`).
- **ARM Variabile Privato** — un prestito variabile al 4,75% con un tetto dell'1% per adeguamento / 5% sulla vita del prestito (`CAPS=`), un `MAXRATE=` del 9,75%, `ADJUSTFREQ=` annuale e la parola chiave `WORSTCASE`.
- **Riduzione Tasso 2-1 del Datore di Lavoro** — un avvio sovvenzionato al 4,50% che, secondo il piano dichiarato, sale tramite `BDRATES=` al 5,50% nel secondo anno e al 6,50% in seguito.

L'istruzione `COMPARE` richiede la vista incrociata tra prestiti a 36, 60 e 120 mesi, con un `TAXRATE=` del 22% e un tasso di rendimento minimo attraente del 4% (`MARR=`); `OUTSUM=` e `OUTCOMP=` catturano il riepilogo per prestito e le righe di confronto. Il listato seguente riporta, per ciascun piano e punto di controllo, gli **interessi cumulativi pagati, il capitale cumulativo estinto e il saldo residuo** — una chiara visione dell'avanzamento dell'ammortamento tra i candidati.

> **Nota sugli adeguamenti di tasso.** `PROC LOAN` di Jenner ammortizza ogni piano al suo tasso **iniziale** dichiarato, quindi le impostazioni `CAPS`/`MAXRATE`/`WORSTCASE` dell'ARM e i gradini `BDRATES` della riduzione tasso vengono riportati nel listato come condizioni del prestito ma **non** vengono applicati al calcolo della rata — le cifre dell'ARM e della riduzione tasso qui sotto riflettono i loro tassi iniziali del 4,75% e del 4,50% mantenuti costanti per l'intera durata. Considerate questi due totali come costi nel migliore dei casi (tasso iniziale), non come massimali nel peggiore dei casi.

> **Nota di riconciliazione (verificata in questa esecuzione).** Con la build attuale del motore, il blocco di riepilogo del singolo Prestito #3 (ARM) sotto "Number of loans evaluated" riporta ora un interesse totale di **$11.493,09** (pagamento totale $38.993,09), diverso dal valore di **$7.099,76** al checkpoint di 120 mesi nella tabella `COMPARE` per lo stesso prestito — un'incoerenza interna nella gestione di `WORSTCASE` tra il riepilogo per singolo prestito e la tabella `COMPARE`. Questa incoerenza si riproduce identica anche sul notebook sorgente in inglese non modificato, quindi non è introdotta dalla localizzazione; la tabella `COMPARE` resta coerente con quanto documentato sopra (tassi mantenuti costanti), mentre il blocco di riepilogo del Prestito #3 no — la narrazione seguente cita i valori della tabella `COMPARE`, che sono quelli effettivamente analizzati riga per riga.

In [3]:
PROCEDURA loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         label='Federale Diretto Non Sovvenzionato (Standard)';

   fixed rate=5.99 init=412.50
         label='Rifinanziamento Privato (con commissione)';

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       label='ARM Variabile Privato (scenario peggiore)';

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           label='Riduzione Tasso 2-1 poi 6,5%';

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
ESEGUIRE;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Federale Diretto Non Sovvenzionato (Standard)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Rifinanziamento Privato (con commissione)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: ARM Variabile Privato (scenario peggiore)
    Loan Type:                   Adjustable Rate
 


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Passo 4 — Rieseguire da solo il piano fisso raccomandato

Per il mutuatario che dà valore alla certezza della rata, il piano federale a tasso fisso standard è la scelta sicura predefinita. Lo rieseguiamo isolatamente per confermarne l'economia principale: la stessa rata mensile di **$312,68**, il totale pagato di **$37.521,22** e gli interessi totali di **$10.021,22** visti nel confronto tra quattro piani, ora presentati come riepilogo di prestito autonomo.

In [4]:
PROCEDURA loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed label='Federale Diretto Non Sovvenzionato (Standard)';
ESEGUIRE;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Federale Diretto Non Sovvenzionato (Standard)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Federale Diretto Non Sovvenzionato (Standard)
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Interpretazione dei risultati

Tutti e quattro i piani ammortizzano completamente lo stesso capitale di $27.500 in 120 mesi (il saldo residuo di ogni piano raggiunge $0,00 al mese 120 nella tabella COMPARE), quindi la decisione dipende dalla **rata mensile** e dagli **interessi totali al tasso dichiarato**:

| Piano | Tasso | Rata | Interessi totali | Note |
|------|------|------|------|-------|
| Federale Diretto Standard | 6,53% | $312,68 | $10.021,22 | Nessuna commissione; protezioni più forti per il mutuatario |
| Rifinanziamento Privato | 5,99% | $305,17 | $9.120,20 | Commissione di attivazione di $412,50 |
| ARM Variabile Privato | 4,75% (iniziale) | $288,33 | $7.099,76 | Il tasso può salire; la cifra è il costo al tasso iniziale |
| Riduzione Tasso 2-1 del Datore di Lavoro | 4,50% (iniziale) | $285,01 | $6.700,67 | Dipende dalla continuità del rapporto di lavoro |

- Il piano **federale standard** è il più costoso in termini di interessi ($10.021,22) ma offre una rata fissa e prevedibile di $312,68 senza commissione.
- Il **rifinanziamento privato** riduce il tasso al 5,99% ($9.120,20 di interessi, $901 in meno rispetto al piano federale) ma addebita una commissione di attivazione di $412,50, quindi il suo vantaggio netto rispetto al piano federale è di circa $488 di interessi meno la commissione — significativo solo se il prestito dura abbastanza a lungo da non essere rifinanziato.
- L'**ARM** e la **riduzione tasso** mostrano qui gli interessi più bassi ($7.099,76 e $6.700,67) unicamente perché i loro tassi **iniziali** sono ben al di sotto delle offerte a tasso fisso. Come notato nel Passo 3, Jenner mantiene costanti questi tassi iniziali, quindi queste sono cifre nel migliore dei casi: un ARM reale che si adegua al rialzo — o una riduzione tasso che perde la sovvenzione del datore di lavoro — porterebbe a cifre più alte, e la rata non è garantita.

La tabella `COMPARE` chiarisce ulteriormente questo aspetto mostrando quanto velocemente si ammortizza ciascun piano. A 36 mesi il piano federale ha pagato $4.792,27 di interessi ed estinto $6.464,10 di capitale (saldo $21.035,90), mentre la riduzione tasso ha pagato solo $3.263,97 di interessi ed estinto $6.996,24 di capitale (saldo $20.503,76) — i piani a tasso più basso costano meno interessi e riducono il capitale più rapidamente nei primi tre anni. Per un neolaureato avverso al rischio, la certezza del tasso del piano federale standard giustifica spesso i suoi interessi più alti; per un mutuatario fiducioso in un impiego stabile e duraturo, la riduzione tasso sovvenzionata offre il massimo risparmio tra le opzioni che fissano davvero il tasso.
- **Nota tecnica.** In questa esecuzione, il blocco di riepilogo standalone del Prestito #3 (ARM) riporta $11.493,09 di interessi totali anziché $7.099,76 — un disallineamento tra il riepilogo per singolo prestito e la tabella `COMPARE` (che invece resta a $7.099,76 al checkpoint di 120 mesi, coerente con i valori a 36 e 60 mesi citati sopra). Il disallineamento è riproducibile anche sul notebook inglese non modificato: è un'incoerenza nota del motore PROC LOAN e non un effetto della traduzione.